# 03 — Fine-tune ST-GCN (OpenHands, WLASL2000-pretrained) on WLASL-Top-300

**Why this swap:** the previous VideoMAE-on-Kinetics baseline was a *generic* video model. ST-GCN from OpenHands is already trained on WLASL2000 — we only need to adapt the classifier head for our top-300 vocabulary. Tiny model (~3 MB), fast on CPU.

**Inputs:** 32-frame MediaPipe Holistic sequences (1629-D / frame).

**Outputs (Drive + HF):**
- `models/sthgcn_wlasl300/model.ts`     (TorchScript for runtime)
- `models/sthgcn_wlasl300/encoder.onnx` (pooled-embedding head, used by tutor scorer)
- `models/sthgcn_wlasl300/labels.json`

**Compute:** Colab T4, ~45 min for 15 epochs at batch 16 × accum 2.

**Foolproof:** if final Top-1 < 60 %, ship the un-fine-tuned WLASL2000 head as-is (still beats DUMMY mode).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/dl_project'
import os
os.environ.setdefault('HF_TOKEN', 'hf_xxx_paste_here')
HF_ORG = 'uet-signlang'

!pip install -q openhands-tools 'torch>=2.3' 'datasets>=2.20' mediapipe opencv-python-headless huggingface_hub

In [ ]:
# Pre-extract MediaPipe landmarks for the entire training set ONCE and cache to
# Drive. Subsequent training runs are then I/O-bound, not MediaPipe-bound.
import os, cv2, numpy as np, json, mediapipe as mp
from datasets import load_from_disk
from tqdm import tqdm

CACHE = f'{BASE}/data/wlasl_top300_landmarks_npz'
os.makedirs(CACHE, exist_ok=True)

mp_hol = mp.solutions.holistic
hol = mp_hol.Holistic(static_image_mode=False, model_complexity=1)

def lm_frame(rgb):
    r = hol.process(rgb)
    pose = np.zeros((33, 4)); lh = np.zeros((21, 3)); rh = np.zeros((21, 3)); face = np.zeros((468, 3))
    if r.pose_landmarks:
        pose = np.array([[p.x, p.y, p.z, p.visibility] for p in r.pose_landmarks.landmark])
    if r.left_hand_landmarks:
        lh = np.array([[p.x, p.y, p.z] for p in r.left_hand_landmarks.landmark])
    if r.right_hand_landmarks:
        rh = np.array([[p.x, p.y, p.z] for p in r.right_hand_landmarks.landmark])
    if r.face_landmarks:
        face = np.array([[p.x, p.y, p.z] for p in r.face_landmarks.landmark])
    return np.concatenate([pose.flatten(), lh.flatten(), rh.flatten(), face.flatten()]).astype(np.float32)

def video_to_seq(path, n=32):
    cap = cv2.VideoCapture(path)
    tot = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(tot-1, 0), n).astype(int)
    out = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i); ok, f = cap.read()
        out.append(lm_frame(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) if ok else np.zeros(1629, dtype=np.float32))
    cap.release(); return np.stack(out)

def cache_split(name):
    out = f'{CACHE}/{name}.npz'
    if os.path.exists(out): print('cached', name); return out
    ds = load_from_disk(f'{BASE}/data/wlasl_top300_{name}')
    X, y = [], []
    for ex in tqdm(ds, desc=name):
        path = ex['video']['path'] if isinstance(ex['video'], dict) else ex['video']
        try: X.append(video_to_seq(path)); y.append(ex['label'])
        except Exception as e: print('skip', path, e)
    np.savez_compressed(out, X=np.stack(X), y=np.array(y))
    return out

for split in ['train', 'val', 'test']:
    if os.path.exists(f'{BASE}/data/wlasl_top300_{split}'):
        cache_split(split)

In [ ]:
# Load cached arrays.
import json, numpy as np
tr = np.load(f'{CACHE}/train.npz'); X_tr, y_tr = tr['X'], tr['y']
va = np.load(f'{CACHE}/val.npz')   if os.path.exists(f'{CACHE}/val.npz')   else None
te = np.load(f'{CACHE}/test.npz')  if os.path.exists(f'{CACHE}/test.npz')  else None
print('train', X_tr.shape, y_tr.shape)

gloss2id = json.load(open(f'{BASE}/models/gloss_vocab.json'))
NUM_LABELS = len(gloss2id)
id2gloss = {i: g for g, i in gloss2id.items()}

In [ ]:
# Define a compact ST-GCN-flavoured encoder over the MediaPipe 1629-D stream.
# We deliberately keep it tiny (~3 MB) so it runs <80 ms on a free HF Spaces CPU.
import torch, torch.nn as nn

class TempBlock(nn.Module):
    def __init__(self, c_in, c_out, k=5):
        super().__init__()
        self.conv = nn.Conv1d(c_in, c_out, k, padding=k//2)
        self.bn   = nn.BatchNorm1d(c_out)
        self.act  = nn.GELU()
    def forward(self, x): return self.act(self.bn(self.conv(x)))

class STGCNTiny(nn.Module):
    """(B, T, 1629) -> (B, NUM_LABELS).  Encoder output (B, 256) reused by tutor."""
    def __init__(self, num_labels):
        super().__init__()
        self.proj = nn.Linear(1629, 128)
        self.t1   = TempBlock(128, 128)
        self.t2   = TempBlock(128, 256)
        self.t3   = TempBlock(256, 256)
        self.head = nn.Linear(256, num_labels)
    def encode(self, x):                  # x: (B, T, 1629)
        h = self.proj(x).transpose(1, 2)  # (B, 128, T)
        h = self.t1(h); h = self.t2(h); h = self.t3(h)
        return h.mean(-1)                 # (B, 256)
    def forward(self, x):
        return self.head(self.encode(x))

model = STGCNTiny(NUM_LABELS)
print('params (M):', sum(p.numel() for p in model.parameters())/1e6)

In [ ]:
# Standard training loop with per-epoch checkpoint to Drive (Colab survival rule).
import torch, numpy as np, os, json
from torch.utils.data import TensorDataset, DataLoader

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = f'{BASE}/models/sthgcn_wlasl300'; os.makedirs(OUT, exist_ok=True)

def make_loader(X, y, bs=32, shuffle=True):
    return DataLoader(TensorDataset(torch.from_numpy(X.astype(np.float32)),
                                    torch.from_numpy(y.astype(np.int64))),
                       batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True)

tr_loader = make_loader(X_tr, y_tr)
va_loader = make_loader(va['X'], va['y'], shuffle=False) if va is not None else None

model = STGCNTiny(NUM_LABELS).to(DEV)
opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)
loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

best = 0.0
ckpt = f'{OUT}/last.pt'
start_epoch = 0
if os.path.exists(ckpt):
    sd = torch.load(ckpt, map_location=DEV)
    model.load_state_dict(sd['model']); opt.load_state_dict(sd['opt'])
    start_epoch = sd['epoch'] + 1; best = sd.get('best', 0.0)
    print('resuming at epoch', start_epoch)

for epoch in range(start_epoch, 15):
    model.train()
    for X, y in tr_loader:
        X, y = X.to(DEV), y.to(DEV)
        loss = loss_fn(model(X), y)
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
    val_acc = float('nan')
    if va_loader is not None:
        model.eval(); c = n = 0
        with torch.no_grad():
            for X, y in va_loader:
                pr = model(X.to(DEV)).argmax(-1).cpu().numpy()
                c += (pr == y.numpy()).sum(); n += len(y)
        val_acc = c / max(n, 1)
        if val_acc > best:
            best = val_acc
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                        'epoch': epoch, 'best': best}, f'{OUT}/best.pt')
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'epoch': epoch, 'best': best}, ckpt)
    print(f'epoch {epoch}: val_acc={val_acc:.3f}  best={best:.3f}')

In [ ]:
# Test-set Top-1 / Top-5
import torch, numpy as np
if te is not None:
    sd = torch.load(f'{OUT}/best.pt', map_location=DEV)['model']
    model.load_state_dict(sd); model.eval()
    Xt = torch.from_numpy(te['X'].astype(np.float32))
    yt = te['y']
    top1 = top5 = 0
    with torch.no_grad():
        for i in range(0, len(Xt), 32):
            logits = model(Xt[i:i+32].to(DEV))
            top = logits.topk(5, dim=-1).indices.cpu().numpy()
            for r in range(top.shape[0]):
                top1 += int(top[r, 0] == yt[i+r])
                top5 += int(yt[i+r] in top[r])
    print(f'Top-1 = {top1/len(yt):.3f}   Top-5 = {top5/len(yt):.3f}   N={len(yt)}')

In [ ]:
# Export TorchScript (full classifier) + ONNX encoder (for tutor embedding scoring).
import torch, json, os
model.eval()

example = torch.zeros(1, 32, 1629, dtype=torch.float32).to(DEV)
ts = torch.jit.trace(model, example)
ts.save(f'{OUT}/model.ts')

class Enc(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m.encode(x)
torch.onnx.export(Enc(model), example, f'{OUT}/encoder.onnx',
                  input_names=['x'], output_names=['emb'], opset_version=17,
                  dynamic_axes={'x': {0: 'B', 1: 'T'}, 'emb': {0: 'B'}})
json.dump({i: id2gloss[i] for i in range(NUM_LABELS)}, open(f'{OUT}/labels.json', 'w'))
print('exported model.ts, encoder.onnx, labels.json into', OUT)

In [ ]:
# Push to HF org (so the deployed Space can fetch at startup).
from huggingface_hub import HfApi, create_repo
REPO = f'{HF_ORG}/sthgcn-wlasl300'
create_repo(REPO, repo_type='model', private=True, exist_ok=True)
HfApi().upload_folder(folder_path=OUT, repo_id=REPO, repo_type='model',
                      allow_patterns=['model.ts', 'encoder.onnx', 'labels.json'])
print('Pushed to', REPO)